# Kafka Producer — 模拟实时出租车数据流

读取批处理输出的 Parquet → 逐条发送到 Kafka topic `tlc-trips`，模拟出租车实时叫车数据流。

真实场景中，这个 Producer 对应的是出租车计费系统实时上报行程数据。

## 1. 导入依赖

In [1]:
import json
import time
import os
import pandas as pd
from kafka import KafkaProducer
from datetime import datetime

print('依赖导入成功')

依赖导入成功


## 2. 初始化 Kafka Producer

In [2]:
producer = KafkaProducer(
    bootstrap_servers='localhost:9092',
    value_serializer=lambda v: json.dumps(v, default=str).encode('utf-8'),
    key_serializer=lambda k: str(k).encode('utf-8')
)

TOPIC = 'tlc-trips'
print(f'Producer 连接成功，目标 topic: {TOPIC}')

Producer 连接成功，目标 topic: tlc-trips


## 3. 读取处理好的 Parquet 数据

In [3]:
# 只读 1 月数据做演示，足够展示流式效果
PROCESSED_PATH = os.path.expanduser(
    '~/tlc-spark-pipeline/data/processed/trips/pickup_year=2024/pickup_month=1'
)

df = pd.read_parquet(PROCESSED_PATH)

# 按上车时间排序，模拟真实时序
df = df.sort_values('tpep_pickup_datetime').reset_index(drop=True)

print(f'读取记录数: {len(df):,}')
print(f'时间范围: {df["tpep_pickup_datetime"].min()} ~ {df["tpep_pickup_datetime"].max()}')
df[['tpep_pickup_datetime', 'PULocationID', 'DOLocationID', 'fare_amount', 'time_of_day']].head()

读取记录数: 2,713,403
时间范围: 2024-01-01 00:00:00 ~ 2024-01-31 23:59:55


,tpep_pickup_datetime,PULocationID,DOLocationID,fare_amount,time_of_day
0,2024-01-01 00:00:00,140,140,4.4,late_night
1,2024-01-01 00:00:02,161,236,8.6,late_night
2,2024-01-01 00:00:03,136,18,5.1,late_night
3,2024-01-01 00:00:04,125,157,33.8,late_night
4,2024-01-01 00:00:05,132,41,70.0,late_night


## 4. 发送数据到 Kafka

In [4]:
SEND_LIMIT = 5000
DELAY_SECONDS = 0.02   # 每条间隔 20ms，5000 条约 100 秒发完

sent = 0
errors = 0

for idx, row in df.head(SEND_LIMIT).iterrows():
    record = {
        'VendorID':             int(row['VendorID']) if pd.notna(row['VendorID']) else None,
        'pickup_datetime':      str(row['tpep_pickup_datetime']),
        'dropoff_datetime':     str(row['tpep_dropoff_datetime']),
        'PULocationID':         int(row['PULocationID']),
        'DOLocationID':         int(row['DOLocationID']),
        'trip_distance':        float(row['trip_distance']),
        'fare_amount':          float(row['fare_amount']),
        'tip_amount':           float(row['tip_amount']),
        'total_amount':         float(row['total_amount']),
        'trip_duration_min':    float(row['trip_duration_min']),
        'avg_speed_mph':        float(row['avg_speed_mph']),
        'time_of_day':          str(row['time_of_day']),
        'payment_label':        str(row['payment_label']),
        'event_timestamp':      str(row['tpep_pickup_datetime'])   # 与 pickup_datetime 保持一致
    }

    try:
        # key 用 PULocationID，保证同一上车区域的消息落到同一 partition
        producer.send(TOPIC, key=record['PULocationID'], value=record)
        sent += 1

        if sent % 500 == 0:
            print(f'已发送 {sent} 条 | 最新: {record["pickup_datetime"]} | 区域 {record["PULocationID"]} → {record["DOLocationID"]} | ${record["fare_amount"]}')

        time.sleep(DELAY_SECONDS)

    except Exception as e:
        errors += 1
        print(f'发送失败: {e}')

producer.flush()
print(f'\n完成！成功发送 {sent} 条，失败 {errors} 条')

已发送 500 条 | 最新: 2024-01-01 00:10:30 | 区域 246 → 79 | $17.0


已发送 1000 条 | 最新: 2024-01-01 00:16:19 | 区域 137 → 232 | $12.1


已发送 1500 条 | 最新: 2024-01-01 00:21:39 | 区域 43 → 238 | $7.9


已发送 2000 条 | 最新: 2024-01-01 00:26:31 | 区域 249 → 261 | $15.6


已发送 2500 条 | 最新: 2024-01-01 00:31:27 | 区域 232 → 170 | $17.0


已发送 3000 条 | 最新: 2024-01-01 00:36:30 | 区域 246 → 170 | $29.6


已发送 3500 条 | 最新: 2024-01-01 00:41:12 | 区域 148 → 170 | $17.0


已发送 4000 条 | 最新: 2024-01-01 00:45:54 | 区域 231 → 113 | $9.3


已发送 4500 条 | 最新: 2024-01-01 00:51:09 | 区域 164 → 249 | $11.4


已发送 5000 条 | 最新: 2024-01-01 00:56:22 | 区域 164 → 224 | $10.0

完成！成功发送 5000 条，失败 0 条


## 5. 验证 — 从 Kafka 读回几条消息

In [5]:
from kafka import KafkaConsumer

consumer = KafkaConsumer(
    TOPIC,
    bootstrap_servers='localhost:9092',
    auto_offset_reset='earliest',      # 从最早的消息开始读
    enable_auto_commit=False,
    value_deserializer=lambda m: json.loads(m.decode('utf-8')),
    consumer_timeout_ms=3000           # 3秒没有新消息就停止
)

print('读回前 5 条消息验证：\n')
for i, msg in enumerate(consumer):
    if i >= 5:
        break
    v = msg.value
    print(f'Partition {msg.partition} | Offset {msg.offset}')
    print(f'  {v["pickup_datetime"]}  区域 {v["PULocationID"]}→{v["DOLocationID"]}  ${v["fare_amount"]}  {v["time_of_day"]}')
    print()

consumer.close()
producer.close()

读回前 5 条消息验证：



/var/folders/r1/85vrv9s95n3cdht963956czw0000gn/T/ipykernel_4881/289090040.py:3: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


Partition 1 | Offset 0
  2024-01-01 00:00:03  区域 136→18  $5.1  late_night

Partition 1 | Offset 1
  2024-01-01 00:00:11  区域 113→234  $16.3  late_night

Partition 1 | Offset 2
  2024-01-01 00:01:22  区域 170→162  $5.1  late_night

Partition 1 | Offset 3
  2024-01-01 00:01:58  区域 239→142  $5.1  late_night

Partition 1 | Offset 4
  2024-01-01 00:02:16  区域 142→24  $12.1  late_night

